# ECE1508: Deep Generative Models -- Summer 2026
## Assignment 5: Diffusion Models
## Question 5: DDPM on MNIST Digit 8

In this question, we train a tiny __DDPM__ to generate digit 8 from MNIST.

### Loading Modules

In [ ]:
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torchvision.utils import make_grid
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from tqdm import tqdm


# Device
if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print('Using device:', device)

Using device: mps


### DDPM Setup

In DDPM, we first choose a $\beta$-schedule, and then derive the other quantities from it. Recall that

$$
\beta_t \in (0,1)
$$

represents the amount of noise added at step $t$. We consider a linear scheduling between a `beta_start` and `beta_end`. This means the noise level gradually increases from `beta_start` to `beta_end` over `T` timesteps.

Once the $\beta_t$ is defined, we compute

$$
\alpha_t = 1 - \beta_t
$$

which gives the amount of signal that remains after one diffusion step. We also define the cumulative product of $\alpha$ as

$$
\bar{\alpha}_t = \prod*{s=1}^{t} \alpha_s.
$$

This tells us how much of the original image remains after $t$ noising steps. The forward diffusion equation is then given by

$$
x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon
$$

where $\epsilon$ is standard Gaussian noise. The reverse diffusion step uses the posterior which is given by

$$
q(x_{t-1} \mid x_t, x_0)
$$

whose variance is

$$
\tilde{\beta}_t = \beta_t \frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}
$$

In the sequel, we set the parameters needed for forward and backward diffusion.

In [ ]:
# Process Length
T = 100 
beta_start = 1e-4
beta_end = 2e-2

# Set Beta linearly increasing
betas = ## COMPLETE ##

# Compute alpha = 1 - beta
alphas = ## COMPLETE ##
alpha_bars = ## COMPLETE ##

# Compute all parameters
sqrt_alphas = ## COMPLETE ##
sqrt_alpha_bars = ## COMPLETE ##
sqrt_one_minus_alpha_bars = ## COMPLETE ##
sqrt_recip_alphas = t## COMPLETE ##
posterior_variance = ## COMPLETE ##

# Let's define the extract function
def extract(a, t, x_shape):
    out = a.gather(0, t)
    return out.view(-1, 1, 1, 1).expand(x_shape)

We also use the same sinusoidal time embedding as the one in Question 2

In [ ]:
def sinusoidal_embedding(t, dim=32):
    ## COMPLETE ##

### Denoising Network

We now implement a simple ResNet for denoising. This is not going to work very sophisticated, but it can still do the job to some basic extent. In this model, the residual block is considered as follows:

Let $x$ be the input map, and $\texttt{emb}$ be the time embedding. Then, the block computes

$$
h \gets \mathrm{Conv}\Big(\mathrm{SiLU}(\mathrm{GN}(x))\Big)
$$
with a  a $3 \times 3$ filter, where $(\mathrm{GN})$ is Group Normalization. It then incorporate time embedding as
$$
h \gets h + W_t \texttt{emb}
$$
for some linear layer $W_t$. We then pass another $3\times 3$ convolution and compute
$$
h \gets \mathrm{Conv} \Big(\mathrm{SiLU}(\mathrm{GN}(h))\Big).
$$
The output is then given as
$$
y = h + \mathrm{Skip}(x)
$$
where $\mathrm{Skip}(x)$ is
* either identity if input and output dimensions match
* or a $1\times 1$ convolution if they do not match. 

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_ch=64):
        super().__init__()
        ## COMPLETE ##

    def forward(self, x, temb):
        ## COMPLETE ##

We now implement the Denoiser Model as follows

1. **Time Embedding**  
   timestep $t$ is converted into a sinusoidal embedding, then passed through an MLP to get a 64-dim time vector.

2. **Input Preparation**  
   the input image $x$ is first mapped to 32 channels by a $3 \times 3$ convolution.

3. **Residual Processing**  
   three ResBlocks process the feature map as
   $$
   32 \rightarrow 32 \rightarrow 64 \rightarrow 64
   $$
   and __each block receives the time embedding.__

4. **Output Head**  
   Output head applies
   
   GroupNorm + SiLU + final $3 \times 3$ convolution 
   
   to map the features back to one output channel.


In [ ]:
class Denoiser(nn.Module):
    def __init__(self, time_dim=32):
        super().__init__()
        self.time_dim = time_dim

        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
        )

        ## COMPLETE ##

    def forward(self, x, t):
        # t: integer timestep tensor in [0, T-1]
        ## COMPLETE ##

### Forward Diffusion and Loss

For each minibatch we sample a random time step, corrupt the clean image, and train
the network to predict the added noise.

In [ ]:
# Let us implement the time sampling and image corruption
def q_sample(x0, t, noise=None):
    ## COMPLETE ##


## Define the loss between the model noise prediction and true noise
def ddpm_loss(model, x0):
    ## COMPLETE ##

### Reverse Diffusion

Finally, we implement the time trajectory that starts from pure Gaussian noise and iteratively denoises for `T` steps.

At a single time step 

$$
x_{t−1} = \mu_{\theta} (x,t) + \sqrt{\tilde{\beta}_t} z
$$

with $z \sim N(0,1)$, except at the final step $t=0$, where $z$ is skipped.

In [ ]:
# Sample in a single time
@torch.no_grad()
def p_sample(model, x, t):
   ## COMPLETE ##


# Go back in time
@torch.no_grad()
def sample_ddpm(model, shape):
   ## COMPLETE ##


# Show the output
def show_images(x, nrow=4, title=None):
    x = x.detach().cpu().clamp(-1, 1)
    x = (x + 1) / 2
    grid = make_grid(x, nrow=nrow)
    plt.figure(figsize=(5, 5))
    if title is not None:
        plt.title(title)
    plt.axis('off')
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
    plt.show()

### Data

We keep working with digit `8`. You can increase the subset size later if you want slightly better samples.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

full_train = MNIST(root='data', train=True, download=True, transform=transform)
indices_8 = [i for i, (_, y) in enumerate(full_train) if y == 8]
subset_size = min(3000, len(indices_8))
train_indices = indices_8[:subset_size]
train_set = Subset(full_train, train_indices)
train_loader = ## COMPLETE ##


### Training
We now write the training loop.

In [ ]:
model = Denoiser(time_dim=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 100
loss_history = []

for epoch in range(num_epochs):
    model.train()
    epoch_losses = []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
    for x, _ in pbar:
        ## COMPLETE ##

    print(f'Epoch {epoch+1}: loss = {avg_loss:.4f}')

### Sampling
Let us now sample the model and look at the outputs images. 

In [ ]:
samples = ## COMPLETE ##
show_images(samples, nrow=4, title='Generated MNIST 8s')

# Plot also the training curve
plt.figure(figsize=(5, 3))
## COMPLETE ##
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.title('DDPM training curve')
plt.show()

### Question: _Track the samples over time and explain your observation._
_## COMPLETE ##_

### Question: _Compared to VAE and GAN, do you think that it was a good idea to use DDPM for this simple task? Explain your answer._
_## COMPLETE ##_